# Step 0: Import files

In [2]:
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

import numpy as np
import nba_api
import nba_api.stats.endpoints as ep
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.endpoints import PlayerEstimatedMetrics
from nba_api.stats.static import teams

df_team_stats = pd.read_csv("TeamStatistics.csv")
df_team_stats_ext = pd.read_csv("TeamStatisticsExtended.csv")

# Assign Columns

In [3]:
metrics = PlayerEstimatedMetrics(season='2025-26')
df = metrics.get_data_frames()[0]
print(df.columns)

colToPrint = [ 'PLAYER_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 
              'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT',
              'E_USG_PCT', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
]

rankColumns = ['GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
       'E_OFF_RATING_RANK', 'E_DEF_RATING_RANK', 'E_NET_RATING_RANK',
       'E_AST_RATIO_RANK', 'E_OREB_PCT_RANK', 'E_DREB_PCT_RANK',
       'E_REB_PCT_RANK', 'E_TOV_PCT_RANK', 'E_USG_PCT_RANK', 'E_PACE_RANK']


col_Altogether = [ 
    'Win',
    '3PM', '%Points_3PT', 'FTA', 'diff_FTM',
    '%Points_2PT', '%Points_2PT_MidRange',
    "diff_OReb",
    'DReb', 'OppPointsSecondChance',
    'diff_%Points_InPaint', 'diff_OppPointsInPaint',
    'OppEffFG%', 'diff_PointsOffTO', 'OpponentScore', 'OppTO%', 'OppFTA_Rate', 'PF', 
    'OppPointsFastBreak',
    'Ast', 
    'TO', '%Assisted_FGM',
]

Index(['PLAYER_ID', 'PLAYER_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN',
       'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_AST_RATIO',
       'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TOV_PCT', 'E_USG_PCT',
       'E_PACE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
       'E_OFF_RATING_RANK', 'E_DEF_RATING_RANK', 'E_NET_RATING_RANK',
       'E_AST_RATIO_RANK', 'E_OREB_PCT_RANK', 'E_DREB_PCT_RANK',
       'E_REB_PCT_RANK', 'E_TOV_PCT_RANK', 'E_USG_PCT_RANK', 'E_PACE_RANK'],
      dtype='object')


In [4]:
pd.set_option("display.max_columns", None)
print(df[colToPrint].head(2))

      PLAYER_NAME  GP   W   L  W_PCT   MIN  E_OREB_PCT  E_DREB_PCT  E_REB_PCT  \
0  Bub Carrington  82  17  65  0.207  27.7       0.015       0.117      0.065   
1   Javonte Green  82  60  22  0.732  17.6       0.050       0.133      0.091   

   E_TOV_PCT  E_USG_PCT  GP_RANK  W_RANK  L_RANK  W_PCT_RANK  MIN_RANK  
0     14.025      0.180        1     334     582         523       134  
1      7.895      0.148        1       4     279          65       334  


# All endpoints in API

In [5]:
dir(ep)


['AllTimeLeadersGrids',
 'AssistLeaders',
 'AssistTracker',
 'BoxScoreAdvancedV2',
 'BoxScoreAdvancedV3',
 'BoxScoreDefensiveV2',
 'BoxScoreFourFactorsV2',
 'BoxScoreFourFactorsV3',
 'BoxScoreHustleV2',
 'BoxScoreMatchupsV3',
 'BoxScoreMiscV2',
 'BoxScoreMiscV3',
 'BoxScorePlayerTrackV3',
 'BoxScoreScoringV2',
 'BoxScoreScoringV3',
 'BoxScoreSummaryV2',
 'BoxScoreSummaryV3',
 'BoxScoreTraditionalV2',
 'BoxScoreTraditionalV3',
 'BoxScoreUsageV2',
 'BoxScoreUsageV3',
 'CommonAllPlayers',
 'CommonPlayerInfo',
 'CommonPlayoffSeries',
 'CommonTeamRoster',
 'CommonTeamYears',
 'CumeStatsPlayer',
 'CumeStatsPlayerGames',
 'CumeStatsTeam',
 'CumeStatsTeamGames',
 'DefenseHub',
 'DraftBoard',
 'DraftCombineDrillResults',
 'DraftCombineNonStationaryShooting',
 'DraftCombinePlayerAnthro',
 'DraftCombineSpotShooting',
 'DraftCombineStats',
 'DraftHistory',
 'DunkScoreLeaders',
 'FantasyWidget',
 'FranchiseHistory',
 'FranchiseLeaders',
 'FranchisePlayers',
 'GLAlumBoxScoreSimilarityScore',
 'GameR

# Step 1: Gather Data

## All helper functions and testing to merge the necessasry endpoints

In [7]:
# Example: 2025-26 season, Boston Celtics team_id = 1610612738
seasons_and_teams = [
    ("2025-26", 1610612738),
]
game_ids = {
    # "2023-24": "0022300061",
    # "2024-25": "0022400062",
    "2025-26": "0022500001"
}

team_ids = [
    1610612738
]
seasons = [
    # "2023-24", 
    # "2024-25", 
    "2025-26"
]

seasons_and_types = [
    ("2025-26", "Regular Season"),
]

team_player_ids = [
    (1610612738, 2544),  # Example: Celtics + player ID
]

minutes_min = [5]

no_required_endpoints = []

player_comparisons = [
    (
        [2544, 203507],
        [1629029, 1630162]
    ),
]

player_ids = [
    2544,       # LeBron James
    203507,     # Giannis Antetokounmpo
]

team_players_vs_players = [
    {
        "team_id": 1610612738,

        # 5 Celtics players
        "player_id1": 1630625,   # Banton
        "player_id2": 1629646,   # Bassey
        "player_id3": 1628449,   # Boucher
        "player_id4": 1627759,   # Jaylen Brown
        "player_id5": 1630568,   # Garza

        "vs_team_id": 1610612747,

        # 5 Lakers players
        "vs_player_id1": 1629029,  # Luka Doncic
        "vs_player_id2": 1630551,  # Austin Reaves
        "vs_player_id3": 1630224,  # Quentin Grimes
        "vs_player_id4": 1631114,  # Walker Kessler
        "vs_player_id5": 1630702   # Jaden Hardy
    }
]


def run_season_endpoints(seasons, endpoints):
    """
    Run NBA API endpoints that only require a season.
    """
    for season in seasons:
        print(f"\n=== {season} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)
                obj = endpoint(season=season)
                df = obj.get_data_frames()[0]
                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")
            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_team_endpoints(team_ids, endpoints):
    """
    Run NBA API endpoints that only require a team ID.
    """
    for team_id in team_ids:
        print(f"\n=== TeamID: {team_id} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)

                obj = endpoint(team_id=team_id)
                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_game_endpoints(game_ids, endpoints):
    """
    Run NBA API endpoints that only require a game ID.
    game_ids should be a dictionary:
        {season: game_id}
    """
    for season, gid in game_ids.items():
        print(f"\n=== {season} | GameID: {gid} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)
                obj = endpoint(game_id=gid)
                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")


def run_season_team_endpoints(seasons_and_teams, endpoints):
    """
    Run NBA API endpoints that require both a season and team ID.
    seasons_and_teams should contain:
        [(season, team_id), ...]
    """
    for season, team_id in seasons_and_teams:
        print(f"\n=== {season} | TeamID: {team_id} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)
                obj = endpoint(
                    season=season,
                    team_id=team_id
                )
                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_season_season_type_endpoints(seasons_and_types, endpoints):
    """
    Run NBA API endpoints that require:
        season
        season_type
    seasons_and_types should contain:
        [("2025-26", "Regular Season"), ...]
    """
    for season, season_type in seasons_and_types:
        print(f"\n=== {season} | {season_type} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)
                obj = endpoint(
                    season=season,
                    season_type=season_type
                )
                df = obj.get_data_frames()[0]
                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")
            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_minutes_min_endpoints(minutes_min, endpoints):
    """
    Run NBA API endpoints requiring minutes_min.
    """
    for minutes in minutes_min:
        print(f"\n=== Minutes Min: {minutes} ===")
        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)
                obj = endpoint(minutes_min=minutes)
                df = obj.get_data_frames()[0]
                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")
            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_no_required_endpoints(endpoints):
    """
    Run NBA API endpoints that have no required parameters.
    All endpoint parameters are optional and will use their defaults.
    """
    for name, endpoint in endpoints:
        print(f"\n=== {name} ===")

        try:
            pd.set_option("display.max_columns", None)

            obj = endpoint()
            df = obj.get_data_frames()[0]

            print(df.shape)
            print(f"{name}:\n", df.head(), "\n")

        except Exception as e:
            print(f"{name} failed:", e, "\n")
    
def run_team_player_endpoints(team_player_ids, endpoints):
    """
    Run NBA API endpoints requiring both team_id and player_id.

    team_player_ids should contain:
        [(team_id, player_id), ...]
    """
    for team_id, player_id in team_player_ids:
        print(f"\n=== TeamID: {team_id} | PlayerID: {player_id} ===")

        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)

                obj = endpoint(
                    team_id=team_id,
                    player_id=player_id
                )

                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_player_compare_endpoints(player_comparisons, endpoints):
    """
    Run NBA API endpoints requiring:
        player_id_list
        vs_player_id_list

    player_comparisons should contain:
        [(player_id_list, vs_player_id_list), ...]
    """
    for player_id_list, vs_player_id_list in player_comparisons:

        print(
            f"\n=== Players: {player_id_list} | "
            f"Vs Players: {vs_player_id_list} ==="
        )

        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)

                obj = endpoint(
                    player_id_list=player_id_list,
                    vs_player_id_list=vs_player_id_list
                )

                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_player_endpoints(player_ids, endpoints):
    """
    Run NBA API endpoints that require only a player ID.

    player_ids should contain:
        [player_id, ...]
    """
    for player_id in player_ids:
        print(f"\n=== PlayerID: {player_id} ===")

        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)

                obj = endpoint(
                    player_id=player_id
                )

                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

def run_team_players_vs_players_endpoints(matchups, endpoints):
    """
    Run NBA API endpoints requiring:
        team_id
        player_id1 through player_id5
        vs_team_id
        vs_player_id1 through vs_player_id5

    matchups should contain dictionaries with all required parameters.
    """

    for matchup in matchups:

        print(
            f"\n=== TeamID: {matchup['team_id']} "
            f"vs TeamID: {matchup['vs_team_id']} ==="
        )

        for name, endpoint in endpoints:
            try:
                pd.set_option("display.max_columns", None)

                obj = endpoint(
                    team_id=matchup["team_id"],
                    player_id1=matchup["player_id1"],
                    player_id2=matchup["player_id2"],
                    player_id3=matchup["player_id3"],
                    player_id4=matchup["player_id4"],
                    player_id5=matchup["player_id5"],

                    vs_team_id=matchup["vs_team_id"],
                    vs_player_id1=matchup["vs_player_id1"],
                    vs_player_id2=matchup["vs_player_id2"],
                    vs_player_id3=matchup["vs_player_id3"],
                    vs_player_id4=matchup["vs_player_id4"],
                    vs_player_id5=matchup["vs_player_id5"]
                )

                df = obj.get_data_frames()[0]

                print(df.shape)
                print(f"{name}:\n", df.head(), "\n")

            except Exception as e:
                print(f"{name} failed:", e, "\n")

import inspect
def inspect_endpoints(endpoints):
    """
    Inspect NBA API endpoints.
    Prints the required and optional parameters for each endpoint,
    while also returning a dictionary of endpoint requirements.
    Returns:
        dict:
            {
                "EndpointName": {
                    "required": [...],
                    "optional": [...]
                }
            }
    """
    endpoint_requirements = {}
    for name, endpoint in endpoints:
        signature = inspect.signature(endpoint)
        required = []
        optional = []
        for param_name, param in signature.parameters.items():
            # Ignore internal/request-related parameters
            if param_name in ["proxy", "headers", "timeout", "get_request"]:
                continue
            if param.default is inspect.Parameter.empty:
                required.append(param_name)
            else:
                optional.append(
                    f"{param_name}={param.default!r}"
                )
        endpoint_requirements[name] = {
            "required": required,
            "optional": optional
        }
        # Print results
        print(f"\n{'=' * 60}")
        print(f"ENDPOINT: {name}")
        print(f"{'=' * 60}")
        print("\nREQUIRED:")
        if required:
            for param in required:
                print(f"  - {param}")
        else:
            print("  None")
        print("\nOPTIONAL:")
        if optional:
            for param in optional:
                print(f"  - {param}")
        else:
            print("  None")
    return endpoint_requirements

def run_endpoints_by_requirements(
    endpoint_requirements,
    endpoints,
    seasons,
    game_ids,
    team_ids,
    seasons_and_teams,
    seasons_and_types,
    minutes_min,
    team_player_ids,
    player_comparisons,
    player_ids,
    team_players_vs_players
):
    """
    Automatically run endpoints using the appropriate helper
    based on their required parameters.
    """

    # Separate endpoints into groups
    # Separate endpoints into groups
    no_required_endpoints = []
    season_endpoints = []
    game_endpoints = []
    team_endpoints = []
    player_endpoints = []
    season_team_endpoints = []
    team_player_endpoints = []
    player_compare_endpoints = []
    team_players_vs_players_endpoints = []
    season_type_endpoints = []
    minutes_endpoints = []
    unknown_endpoints = []


    for name, endpoint in endpoints:
        required = endpoint_requirements[name]["required"]
        required_set = set(required)
        
        if required_set == set():
            no_required_endpoints.append((name, endpoint))
        
        elif required_set == {"season"}:
            season_endpoints.append((name, endpoint))

        elif required_set == {"game_id"}:
            game_endpoints.append((name, endpoint))
        
        elif required_set == {"team_id"}:
            team_endpoints.append((name, endpoint))

        elif required_set == {"season", "team_id"}:
            season_team_endpoints.append((name, endpoint))

        elif required_set == {"season", "season_type"}:
            season_type_endpoints.append((name, endpoint))

        elif required_set == {"minutes_min"}:
            minutes_endpoints.append((name, endpoint))
        
        elif required_set == {"team_id", "player_id"}:
            team_player_endpoints.append((name, endpoint))

        elif required_set == {"vs_player_id_list", "player_id_list"}:
            player_compare_endpoints.append((name, endpoint))

        elif required_set == {"player_id"}:
            player_endpoints.append((name, endpoint))
        elif required_set == {
                "team_id",
                "player_id1",
                "player_id2",
                "player_id3",
                "player_id4",
                "player_id5",
                "vs_team_id",
                "vs_player_id1",
                "vs_player_id2",
                "vs_player_id3",
                "vs_player_id4",
                "vs_player_id5"
            }:
                team_players_vs_players_endpoints.append((name, endpoint))

        else:
            unknown_endpoints.append((name, endpoint))

    # Run each group
    if season_endpoints:
        run_season_endpoints(
            seasons,
            season_endpoints
        )

    if game_endpoints:
        run_game_endpoints(
            game_ids,
            game_endpoints
        )

    if team_endpoints:
        run_team_endpoints(
            team_ids,
            team_endpoints
        )

    if season_team_endpoints:
        run_season_team_endpoints(
            seasons_and_teams,
            season_team_endpoints
        )

    if season_type_endpoints:
        run_season_season_type_endpoints(
            seasons_and_types,
            season_type_endpoints
        )

    if minutes_endpoints:
        run_minutes_min_endpoints(
            minutes_min,
            minutes_endpoints
        )
        
    if no_required_endpoints:
        run_no_required_endpoints(
            no_required_endpoints
        )

    if team_player_endpoints:
        run_team_player_endpoints(
            team_player_ids,
            team_player_endpoints
        )

    if player_compare_endpoints:
        run_player_compare_endpoints(
            player_comparisons,
            player_compare_endpoints
        )
    if player_endpoints:
        run_player_endpoints(
            player_ids,
            player_endpoints
        )
    
    if team_players_vs_players_endpoints:
        run_team_players_vs_players_endpoints(
            team_players_vs_players,
            team_players_vs_players_endpoints
        )

    # Tell you about anything that doesn't fit
    if unknown_endpoints:

        print("\n" + "=" * 60)
        print("ENDPOINTS REQUIRING NEW HELPER")
        print("=" * 60)

        for name, endpoint in unknown_endpoints:

            required = endpoint_requirements[name]["required"]

            print(f"\n{name}")
            print(f"Required parameters: {required}")

# Reuirements for each endpoint

## Endpoints I have looked through

In [9]:
endpointsNeeding_season_and_teams = [
    #  Shows PLAYER (or PLAYER_ID) on TeamID team and shows EXP experience, as well as HOW_ACQUIRED
    # Can be useful for trying to figure out rotations and such for model predictions
    ("CommonTeamRoster", ep.CommonTeamRoster), 
]

endpointsNeeding_GameID = [
    ("BoxScoreAdvancedV3", ep.BoxScoreAdvancedV3), # Shows box score advanced stats, probably wont need but you never know
    ("BoxScoreDefensiveV2", ep.BoxScoreDefensiveV2), # Shwos defensive stats
    ("BoxScoreFourFactorsV3", ep.BoxScoreFourFactorsV3), # Gives team stats like oppnent FA, stats etc.
    ("BoxScoreHustleV2", ep.BoxScoreHustleV2), # True hustle stats like loose balls, charges, screen assists, box outs, not useful for this but can be used in the future
    ("BoxScoreMatchupsV3", ep.BoxScoreMatchupsV3), # Matchup stats like FG stuff, and potential assist stuff, not useful for this but possibly in the future
    ("BoxScoreMiscV3", ep.BoxScoreMiscV3), # more box score stats, couldbe useful 
    ("BoxScorePlayerTrackV3", ep.BoxScorePlayerTrackV3), #Probably more useful for player props
    ("BoxScoreScoringV3", ep.BoxScoreScoringV3), # more box score stats, couldbe useful 
    ("BoxScoreSummaryV3", ep.BoxScoreSummaryV3), # The basic box score data , maybe useful
    ("BoxScoreTraditionalV3", ep.BoxScoreTraditionalV3),# The basic box score data , maybe useful, V2 is the same too
    ("BoxScoreUsageV3", ep.BoxScoreUsageV3), # Could use but just more usage related stuff
    ("GameRotation", ep.GameRotation), # Rotation stats, would try to figure out what this is IN_TIME_REAL  OUT_TIME_REAL
    ("InfographicFanDuelPlayer", ep.InfographicFanDuelPlayer), # Fantasy statsheet, can be easier way to see game stats
    ("PlayByPlayV3", ep.PlayByPlayV3), # play by play shot shart location stuff for each game
]

endpointsNeeding_season = [
    ("FantasyWidget", ep.FantasyWidget), # General fantasy info could use for player props
    ("GravityLeaders", ep.GravityLeaders), # Leaders in gravity metrics (spacing impact)
    ("HomePageLeaders", ep.HomePageLeaders), # Team shooting state, so points and FGPCT
    ("LeagueDashLineups", ep.LeagueDashLineups), # Ranks lineups used in a season, could be used for etermining lineups
    ("LeagueDashOppPtShot", ep.LeagueDashOppPtShot), # Just looked like shooting stats for all teams in nba for season
    ("LeagueDashPlayerBioStats", ep.LeagueDashPlayerBioStats), # Could get players cumulative P/R/A stats for season
    ("LeagueDashPlayerClutch", ep.LeagueDashPlayerClutch), # Ranking of player in a season

    ("LeagueDashPlayerPtShot", ep.LeagueDashPlayerPtShot), # Player points by shot type
    ("LeagueDashPlayerShotLocations", ep.LeagueDashPlayerShotLocations), # Player shot location data
    ("LeagueDashPlayerStats", ep.LeagueDashPlayerStats), # Player stats dashboard for the season cumulative
    ("LeagueDashPtDefend", ep.LeagueDashPtDefend), # Player defense vs points
    ("LeagueDashPtStats", ep.LeagueDashPtStats), # Teams that have to travel the most? Not actually 100% could mean running miles, etc.
    ("LeagueDashPtTeamDefend", ep.LeagueDashPtTeamDefend), # Team defense vs points
    ("LeagueDashTeamClutch", ep.LeagueDashTeamClutch), # Team Rankings for stats in a season
    ("LeagueDashTeamPtShot", ep.LeagueDashTeamPtShot), # Team points by shot type
    ("LeagueDashTeamShotLocations", ep.LeagueDashTeamShotLocations), # Team shot location data
    ("LeagueDashTeamStats", ep.LeagueDashTeamStats), # Team stats dashboardfor the season cumulative

    ("LeagueGameLog", ep.LeagueGameLog), # League-wide game logs

    ("LeagueHustleStatsPlayer", ep.LeagueHustleStatsPlayer), # Hustle stats for players for season

    ("LeagueHustleStatsTeam", ep.LeagueHustleStatsTeam), # Hustle stats for teams for season
]

endpointsNeeding_season_and_season_type = [
    ("LeagueGameFinder", ep.LeagueGameFinder), # Find games by filters, doesnt take in anything
]

endpointsNeeding_minutes_min = [
    ("LeagueLineupViz", ep.LeagueLineupViz), # Lineup visualization, gives overall lineups stats rather than rankings
]

endpointsNeeding_team_id = [
    ("LeaguePlayerOnDetails", ep.LeaguePlayerOnDetails), # Player on/off details, can input season too

    # ("TeamDashLineups", ep.TeamDashLineups),
    # ("TeamDashPtPass", ep.TeamDashPtPass),
    # ("TeamDashPtReb", ep.TeamDashPtReb),
    # ("TeamDashPtShots", ep.TeamDashPtShots),
    # ("TeamDashboardByGeneralSplits", ep.TeamDashboardByGeneralSplits),
    # ("TeamDashboardByShootingSplits", ep.TeamDashboardByShootingSplits),
    
]

endpointsNeeding_None = [
    ("LeagueSeasonMatchups", ep.LeagueSeasonMatchups), # Season matchups data
    ("LeagueStandings", ep.LeagueStandings), # League standings
    ("LeagueStandingsV3", ep.LeagueStandingsV3), # Updated league standings (V3)
    ("MatchupsRollup", ep.MatchupsRollup), # Matchup field goal percentages
]

endpointsNeeding_team_and_player = [
    # ("PlayerDashPtPass", ep.PlayerDashPtPass),
]

endpointsNeeding_player_compare = [
    ("PlayerCompare", ep.PlayerCompare), # Comparing two players stats
]

endpoints_ReturningEmptyDF = [
    ("PlayerDashPtPass", ep.PlayerDashPtPass),
    ("PlayerDashPtReb", ep.PlayerDashPtReb),
    ("PlayerDashPtShotDefend", ep.PlayerDashPtShotDefend),
    ("PlayerDashPtShots", ep.PlayerDashPtShots),
    ("ScheduleLeagueV2", ep.ScheduleLeagueV2), # Shows league schedule 
    ("ShotChartLeagueWide", ep.ShotChartLeagueWide), # Give you league average shooting splits for a season
]

endpointsNeeding_player_id = [
    ("PlayerDashboardByClutch", ep.PlayerDashboardByClutch), # Good general dashport showing cumulative stats for a season and fantasy stats
    ("PlayerDashboardByShootingSplits", ep.PlayerDashboardByShootingSplits), # Shooting splits and rank for each player
    ("PlayerEstimatedMetrics", ep.PlayerEstimatedMetrics), # Player estimated metrics, only reb%
    ("PlayerGameLog", ep.PlayerGameLog), # Shows player game logs stats
    ("PlayerIndex", ep.PlayerIndex), # Player index
    ("PlayerProfileV2", ep.PlayerProfileV2), # Shows players season stats throughout career
]

endpointsNeeding_team_players_vs_players = [
    ("TeamAndPlayersVsPlayers", ep.TeamAndPlayersVsPlayers), # Compares two lineups together which could be interesting to use, only uses players from two teams, cant be random 5 players
]
# ("ScoreboardV3", ep.ScoreboardV3), Incase but should appearin other ep's

## Remaining endpoints that I haven't taken a look at

In [14]:
# List of endpoints you want to test
endpoints = [
# ("FantasyWidget", ep.FantasyWidget), # General fantasy info could use for player props
#     ("GravityLeaders", ep.GravityLeaders), # Leaders in gravity metrics (spacing impact)
#     ("HomePageLeaders", ep.HomePageLeaders), # Team shooting state, so points and FGPCT
#     ("LeagueDashLineups", ep.LeagueDashLineups), # Ranks lineups used in a season, could be used for etermining lineups
#     ("LeagueDashOppPtShot", ep.LeagueDashOppPtShot), # Just looked like shooting stats for all teams in nba for season
#     ("LeagueDashPlayerBioStats", ep.LeagueDashPlayerBioStats), # Could get players cumulative P/R/A stats for season
#     ("LeagueDashPlayerClutch", ep.LeagueDashPlayerClutch), # Ranking of player in a season

#     ("LeagueDashPlayerPtShot", ep.LeagueDashPlayerPtShot), # Player points by shot type
#     ("LeagueDashPlayerShotLocations", ep.LeagueDashPlayerShotLocations), # Player shot location data
#     ("LeagueDashPlayerStats", ep.LeagueDashPlayerStats), # Player stats dashboard for the season cumulative
#     ("LeagueDashPtDefend", ep.LeagueDashPtDefend), # Player defense vs points
#     ("LeagueDashPtStats", ep.LeagueDashPtStats), # Teams that have to travel the most? Not actually 100% could mean running miles, etc.
#     ("LeagueDashPtTeamDefend", ep.LeagueDashPtTeamDefend), # Team defense vs points
#     ("LeagueDashTeamClutch", ep.LeagueDashTeamClutch), # Team Rankings for stats in a season
#     ("LeagueDashTeamPtShot", ep.LeagueDashTeamPtShot), # Team points by shot type
#     ("LeagueDashTeamShotLocations", ep.LeagueDashTeamShotLocations), # Team shot location data
    # ("LeagueDashTeamStats", ep.LeagueDashTeamStats), # Team stats dashboardfor the season cumulative
        # ("BoxScoreAdvancedV3", ep.BoxScoreAdvancedV3), # Shows box score advanced stats, probably wont need but you never know
        ("BoxScoreTraditionalV3", ep.BoxScoreTraditionalV3)


#     ("LeagueGameLog", ep.LeagueGameLog), # League-wide game logs

#     ("LeagueHustleStatsPlayer", ep.LeagueHustleStatsPlayer), # Hustle stats for players for season

#     ("LeagueHustleStatsTeam", ep.LeagueHustleStatsTeam), # Hustle stats for teams for season
    
    #  ("LeagueDashPtTeamDefend", ep.LeagueDashPtTeamDefend), # Team defense vs points
    # ("LeagueDashTeamClutch", ep.LeagueDashTeamClutch), # Team Rankings for stats in a season
    # ("LeagueDashTeamPtShot", ep.LeagueDashTeamPtShot), # Team points by shot type
    # ("LeagueDashTeamShotLocations", ep.LeagueDashTeamShotLocations), # Team shot location data
    # ("LeagueDashTeamStats", ep.LeagueDashTeamStats), # Team stats dashboardfor the season cumulative

    # ("LeagueDashPlayerStats", ep.LeagueDashPlayerStats), # Player stats dashboard for the season cumulative
    # ("LeagueDashPlayerStats", ep.LeagueDashPlayerStats), # Player stats dashboard for the season cumulative
    # ("LeagueDashPlayerPtShot", ep.LeagueDashPlayerPtShot), # Player points by shot type
    # ("PlayerDashboardByClutch", ep.PlayerDashboardByClutch), # Good general dashport showing cumulative stats for a season and fantasy stats
    # ("PlayerDashboardByClutch", ep.PlayerDashboardByClutch), # Good general dashport showing cumulative stats for a season and fantasy stats
    # ("TeamDetails", ep.TeamDetails),
    # ("TeamEstimatedMetrics", ep.TeamEstimatedMetrics),
    # ("TeamGameLog", ep.TeamGameLog),
    # ("TeamGameLogs", ep.TeamGameLogs),
    # ("TeamGameStreakFinder", ep.TeamGameStreakFinder),
    # ("TeamHistoricalLeaders", ep.TeamHistoricalLeaders),
    # ("TeamInfoCommon", ep.TeamInfoCommon),
    # ("TeamPlayerDashboard", ep.TeamPlayerDashboard),
    # ("TeamPlayerOnOffDetails", ep.TeamPlayerOnOffDetails),
    # ("TeamPlayerOnOffSummary", ep.TeamPlayerOnOffSummary),
    # ("TeamVsPlayer", ep.TeamVsPlayer),
    # ("TeamYearByYearStats", ep.TeamYearByYearStats),

    # ("VideoDetails", ep.VideoDetails),
    # ("VideoDetailsAsset", ep.VideoDetailsAsset),
    # ("VideoEvents", ep.VideoEvents),
    # ("VideoStatus", ep.VideoStatus),

    # ("WinProbabilityPBP", ep.WinProbabilityPBP),

    # # Common endpoints
    # ("AllTimeLeadersGrids", ep.AllTimeLeadersGrids),
    # ("CommonAllPlayers", ep.CommonAllPlayers),
    # ("CommonPlayerInfo", ep.CommonPlayerInfo),
    # ("CommonPlayoffSeries", ep.CommonPlayoffSeries),
    # ("CommonTeamRoster", ep.CommonTeamRoster),
    # ("CommonTeamYears", ep.CommonTeamYears),
    # ("CumeStatsPlayer", ep.CumeStatsPlayer),
    # ("CumeStatsPlayerGames", ep.CumeStatsPlayerGames),
    # ("CumeStatsTeam", ep.CumeStatsTeam),
    # ("CumeStatsTeamGames", ep.CumeStatsTeamGames),
    # ("DefenseHub", ep.DefenseHub),
]


result_dict = inspect_endpoints(endpoints)

result_dict = inspect_endpoints(endpoints)

run_endpoints_by_requirements(
    result_dict,
    endpoints,
    seasons,
    game_ids,
    team_ids,
    seasons_and_teams,
    seasons_and_types,
    minutes_min,
    team_player_ids,
    player_comparisons,
    player_ids,
    team_players_vs_players
)


ENDPOINT: BoxScoreTraditionalV3

REQUIRED:
  - game_id

OPTIONAL:
  - end_period='0'
  - end_range='0'
  - range_type='0'
  - start_period='0'
  - start_range='0'

ENDPOINT: BoxScoreTraditionalV3

REQUIRED:
  - game_id

OPTIONAL:
  - end_period='0'
  - end_range='0'
  - range_type='0'
  - start_period='0'
  - start_range='0'

=== 2025-26 | GameID: 0022500001 ===
(27, 34)
BoxScoreTraditionalV3:
        gameId      teamId       teamCity teamName teamTricode teamSlug  \
0  0022500001  1610612760  Oklahoma City  Thunder         OKC  thunder   
1  0022500001  1610612760  Oklahoma City  Thunder         OKC  thunder   
2  0022500001  1610612760  Oklahoma City  Thunder         OKC  thunder   
3  0022500001  1610612760  Oklahoma City  Thunder         OKC  thunder   
4  0022500001  1610612760  Oklahoma City  Thunder         OKC  thunder   

   personId firstName          familyName                  nameI  \
0   1629652  Luguentz                Dort                L. Dort   
1   1631096      Che

## Debugging

In [54]:
stats = leaguedashplayerstats.LeagueDashPlayerStats(season='2024-25')
df = stats.get_data_frames()[0]
print(df.head())


   PLAYER_ID    PLAYER_NAME NICKNAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP  \
0    1630639    A.J. Lawson     A.J.  1610612761               TOR  24.0  26   
1    1631260       AJ Green       AJ  1610612749               MIL  25.0  73   
2    1642358     AJ Johnson       AJ  1610612764               WAS  20.0  29   
3     203932   Aaron Gordon    Aaron  1610612743               DEN  29.0  51   
4    1628988  Aaron Holiday    Aaron  1610612745               HOU  28.0  62   

    W   L  W_PCT          MIN  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  FTM  \
0  14  12  0.538   486.410000   80  190   0.421    33   101    0.327   43   
1  44  29  0.603  1659.141667  182  424   0.429   155   363    0.427   22   
2   8  21  0.276   638.386667   82  213   0.385    24    90    0.267   32   
3  33  18  0.647  1446.716667  264  497   0.531    75   172    0.436  145   
4  39  23  0.629   792.246667  117  268   0.437    72   181    0.398   34   

   FTA  FT_PCT  OREB  DREB  REB  AST  TOV  STL  BLK  BLK

# Step 2: Creating the Dataframe

#### Get team id's 

In [21]:
nba_teams = teams.get_teams()

# Dictionary: Team ID -> Team Name
team_id_to_name = {
    team["id"]: team["full_name"]
    for team in nba_teams
}

# Print to verify
for team_id, team_name in team_id_to_name.items():
    print(f"{team_id}: {team_name}")

1610612737: Atlanta Hawks
1610612738: Boston Celtics
1610612739: Cleveland Cavaliers
1610612740: New Orleans Pelicans
1610612741: Chicago Bulls
1610612742: Dallas Mavericks
1610612743: Denver Nuggets
1610612744: Golden State Warriors
1610612745: Houston Rockets
1610612746: Los Angeles Clippers
1610612747: Los Angeles Lakers
1610612748: Miami Heat
1610612749: Milwaukee Bucks
1610612750: Minnesota Timberwolves
1610612751: Brooklyn Nets
1610612752: New York Knicks
1610612753: Orlando Magic
1610612754: Indiana Pacers
1610612755: Philadelphia 76ers
1610612756: Phoenix Suns
1610612757: Portland Trail Blazers
1610612758: Sacramento Kings
1610612759: San Antonio Spurs
1610612760: Oklahoma City Thunder
1610612761: Toronto Raptors
1610612762: Utah Jazz
1610612763: Memphis Grizzlies
1610612764: Washington Wizards
1610612765: Detroit Pistons
1610612766: Charlotte Hornets


#### Create df containing players and what team they are on, and how they were aquired

In [22]:
endpoint_name, endpoint = ("CommonTeamRoster", ep.CommonTeamRoster)

roster_columns = [
    "TeamID",
    "TeamName",
    "SEASON",
    "PLAYER",
    "NUM",
    "POSITION",
    "AGE",
    "EXP",
    "PLAYER_ID",
    "HOW_ACQUIRED"
]

all_rosters = []

for team_id in team_id_to_name:

    result = endpoint(
        team_id=team_id,
        season="2024-25"
    )

    roster = result.common_team_roster.get_data_frame()

    # Add team name using the Team ID dictionary
    roster["TeamName"] = team_id_to_name[team_id]

    # Keep only the desired columns
    roster = roster[roster_columns].copy()

    all_rosters.append(roster)

# Combine all 30 team rosters into one DataFrame
df_rosters_2024_25 = pd.concat(
    all_rosters,
    ignore_index=True
)

# Print final DataFrame
print(df_rosters_2024_25)

KeyboardInterrupt: 

#### Now get cumulative year stats for each player

In [283]:
#### Now get cumulative year stats for each player
# you shoulduse use these 3 endpoint and 

# Get transactions and team roster for 2024-25 season
# Generate team using these
# FIll roster with 

# Get 
# Get injuries
# Set to 9 man rotations
# Get average stats for each player in a game

# ============================================================
# LeagueDashPlayerPtShot + PlayerDashboardByClutch
# 2024-25 Season

# 1. LeagueDashPlayerPtShot
endpoint_name, endpoint = (
    "LeagueDashPlayerPtShot",
    ep.LeagueDashPlayerPtShot
)

shot_result = endpoint(
    league_id="00",
    per_mode_simple="Totals",
    season="2024-25",
    season_type_all_star="Regular Season"
)

df_shot = shot_result.league_dash_ptshots.get_data_frame()

print("LeagueDashPlayerPtShot:")
print(df_shot.shape)
print(df_shot.columns.tolist())

# 2. PlayerDashboardByClutch for every player
endpoint_name, endpoint = (
    "PlayerDashboardByClutch",
    ep.PlayerDashboardByClutch
)

clutch_dfs = []

for player_id in df_shot["PLAYER_ID"].unique():

    try:
        result = endpoint(
            player_id=player_id,
            season="2024-25"
        )

        clutch_df = result.overall_player_dashboard.get_data_frame()

        if not clutch_df.empty:

            clutch_df["PLAYER_ID"] = player_id

            clutch_dfs.append(clutch_df)

    except Exception as e:
        print(f"Error for PLAYER_ID {player_id}: {e}")

# 3. Combine all PlayerDashboardByClutch results
df_clutch = pd.concat(
    clutch_dfs,
    ignore_index=True
)

# 4. Rename clutch columns
clutch_columns = [
    col for col in df_clutch.columns
    if col != "PLAYER_ID"
]

df_clutch = df_clutch.rename(
    columns={
        col: f"PlayerDashboardByClutch_{col}"
        for col in clutch_columns
    }
)

# 5. Merge
df_2024_25_player = df_shot.merge(
    df_clutch,
    on="PLAYER_ID",
    how="left"
)

# 6. Print
print("\nFinal DataFrame:")
print(df_2024_25_player)

df_2024_25_player

LeagueDashPlayerPtShot:
(566, 20)
['PLAYER_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'AGE', 'GP', 'G', 'FGA_FREQUENCY', 'FGM', 'FGA', 'FG_PCT', 'EFG_PCT', 'FG2A_FREQUENCY', 'FG2M', 'FG2A', 'FG2_PCT', 'FG3A_FREQUENCY', 'FG3M', 'FG3A', 'FG3_PCT']

Final DataFrame:
     PLAYER_ID              PLAYER_NAME  PLAYER_LAST_TEAM_ID  \
0      1628983  Shai Gilgeous-Alexander           1610612760   
1      1630162          Anthony Edwards           1610612750   
2      1630595          Cade Cunningham           1610612765   
3      1628369             Jayson Tatum           1610612738   
4      1630224              Jalen Green           1610612745   
..         ...                      ...                  ...   
561    1629056            Terence Davis           1610612758   
562    1630678             Terry Taylor           1610612758   
563    1642389              Zyon Pullin           1610612763   
564    1642434              Riley Minix           1610612759   

## Export

In [33]:
df_2024_25_player.to_csv("PlayerStats2024_25.csv", index=False)

temp = pd.read_csv("PlayerStats2024_25.csv")
print(temp)


NameError: name 'df_2024_25_player' is not defined

# Step 3: Read new .csv file

In [24]:
# Fetch per-game player averages for 2024-25 season
stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2024-25',
    season_type_all_star='Regular Season',
    per_mode_detailed='PerGame'
)
df_PlayerStats2425 = stats.get_data_frames()[0]

# Select all numeric columns (drop identifiers/text)
numeric_cols = df_PlayerStats2425.select_dtypes(include=[np.number]).columns.tolist()

# Get unique teams
teams = df_PlayerStats2425['TEAM_ABBREVIATION'].unique()

team_arrays = []
for team in teams:
    team_df = df_PlayerStats2425[df_PlayerStats2425['TEAM_ABBREVIATION'] == team]
    team_array = team_df[numeric_cols].to_numpy()
    team_arrays.append(team_array)

# Pad rosters to equal length
max_players = max(arr.shape[0] for arr in team_arrays)
padded_arrays = [
    np.pad(arr, ((0, max_players - arr.shape[0]), (0, 0)), constant_values=np.nan)
    for arr in team_arrays
]

# Stack into 3D array: [teams, players, stats]
array_3d = np.stack(padded_arrays)

print(array_3d.shape)  # (num_teams, max_players, num_stats)


(30, 23, 66)


In [25]:
print(df_PlayerStats2425.columns.tolist())


['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'FP_HIGH_SCORE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'FP_HIGH_SCORE_RANK', 'TEAM_COUNT']
